# تحلیل جزیره حرارتی شهری (UHI) کلان‌شهر تهران با سنجش از دور

**هدف:** استخراج دمای سطح زمین (LST) از تصاویر Landsat 8/9 Collection 2 Level-2، بررسی رابطهٔ آن با پوشش گیاهی (NDVI) و تراکم ساخت‌وساز (NDBI)، شناسایی کانون‌های داغ و سرد حرارتی با آماره فضایی Getis-Ord Gi*، و مقایسه روند شدت جزیره حرارتی بین تابستان ۲۰۱۵ و ۲۰۲۴.

**داده‌ها:**
- Landsat 8 & 9 Collection 2 Level-2 (بازتاب سطحی + دمای درخشندگی)، از طریق Google Earth Engine
- ESA WorldCover v200 (نقشه پوشش اراضی ۱۰ متری جهانی، ۲۰۲۱)

**روش:** الگوریتم تک‌کاناله (Mono-Window) با تصحیح گسیل‌مندی مبتنی بر NDVI برای استخراج LST؛ نمونه‌برداری شبکه‌ای ۵۰۰ متری؛ همبستگی پیرسون؛ تحلیل خودهمبستگی فضایی Getis-Ord Gi*.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from src.gee_setup import init_earth_engine, load_config, get_aoi
from src.indices import preprocess_collection
from src.lst import build_lst_composite
from src.lulc import load_worldcover
from src.uhi_analysis import (
    sample_grid_to_dataframe, to_geodataframe, correlation_report,
    getis_ord_hotspots, uhi_intensity,
)
from src.visualization import make_interactive_map, plot_lst_histogram, plot_ndvi_lst_scatter, plot_hotspot_map, plot_correlation_heatmap

cfg = load_config()
init_earth_engine(cfg['project']['ee_project_id'])
aoi = get_aoi(cfg)

## ۱. آماده‌سازی تصاویر و استخراج LST برای دو دوره زمانی

In [ ]:
baseline_period = cfg['time_periods']['baseline']
recent_period = cfg['time_periods']['recent']

coll_baseline = preprocess_collection(
    cfg['landsat']['baseline_collection'], aoi,
    baseline_period['start'], baseline_period['end'], cfg['landsat']['cloud_cover_max'])
coll_recent = preprocess_collection(
    cfg['landsat']['recent_collection'], aoi,
    recent_period['start'], recent_period['end'], cfg['landsat']['cloud_cover_max'])

print('تعداد صحنه‌های 2015:', coll_baseline.size().getInfo())
print('تعداد صحنه‌های 2024:', coll_recent.size().getInfo())

lst_baseline = build_lst_composite(coll_baseline)
lst_recent = build_lst_composite(coll_recent)

## ۲. نقشه پوشش اراضی (ESA WorldCover)

In [ ]:
worldcover = load_worldcover(aoi, cfg['lulc']['collection'], cfg['lulc']['year'])

## ۳. نقشه تعاملی (Leaflet) — مقایسه LST دو دوره و NDVI

In [ ]:
m = make_interactive_map(aoi, lst_baseline, lst_recent, worldcover)
m

## ۴. نمونه‌برداری شبکه‌ای و تبدیل به GeoDataFrame

In [ ]:
cell_size = cfg['analysis']['grid_cell_size_m']

df_baseline = sample_grid_to_dataframe(lst_baseline, aoi, cell_size)
gdf_baseline = to_geodataframe(df_baseline)

df_recent = sample_grid_to_dataframe(lst_recent, aoi, cell_size)
gdf_recent = to_geodataframe(df_recent)

gdf_recent.head()

## ۵. توزیع دمایی و روند گرمایش شهری بین دو دوره

In [ ]:
plot_lst_histogram(gdf_baseline, gdf_recent, '../outputs/figures/lst_histogram.png')
warming = gdf_recent['LST_C'].mean() - gdf_baseline['LST_C'].mean()
print(f"میانگین LST 2015: {gdf_baseline['LST_C'].mean():.2f} °C")
print(f"میانگین LST 2024: {gdf_recent['LST_C'].mean():.2f} °C")
print(f"افزایش دما در ۹ سال: {warming:.2f} °C")

## ۶. رابطه دما با پوشش گیاهی و ساخت‌وساز

In [ ]:
corr = correlation_report(gdf_recent)
plot_correlation_heatmap(corr, '../outputs/figures/correlation_heatmap.png')
plot_ndvi_lst_scatter(gdf_recent, '../outputs/figures/ndvi_lst_scatter.png')
corr

## ۷. شناسایی کانون‌های داغ و سرد (Getis-Ord Gi*)

In [ ]:
gdf_hotspots = getis_ord_hotspots(gdf_recent, value_col='LST_C')
plot_hotspot_map(gdf_hotspots, '../outputs/figures/hotspot_map.png')
gdf_hotspots['hotspot_class'].value_counts()

## ۸. نتیجه‌گیری

خروجی‌های این نوت‌بوک (نقشه‌ها، نمودارها و جداول CSV) در پوشه `outputs/` ذخیره می‌شوند و مبنای گزارش نهایی پروژه در `docs/methodology.md` و `README.md` هستند.